# Wake Word "Letícia" para HA Voice PE (v22 — microWakeWord)

## Por que v22 e não continuar o v21?

| | **v21 (openWakeWord)** | **v22 (microWakeWord)** |
|---|---|---|
| Engine | Wyoming add-on (servidor) | ESP32-S3 no chip |
| Compatível com | M5Stack Atom Echo, ESP32-S3-BOX | **HA Voice PE** |
| Output | `leticia.tflite` (Wyoming) | `leticia_mww.tflite` + JSON |
| Deploy | `/share/openwakeword/` | ESPHome YAML + OTA |

O HA Voice PE usa `micro_wake_word` do ESPHome, que roda **direto no ESP32-S3**.
O modelo openWakeWord (v21) é incompatível com esse hardware.

## Instruções
1. **GPU T4 ativa** (Ambiente de execução → Alterar tipo → T4 GPU)
2. Execute **Etapa 1a** → Clique **Reiniciar sessão** quando solicitado
3. Execute **Etapa 1b em diante** (pode usar "Executar tudo a partir daqui")
4. Os arquivos `leticia_mww.tflite` e `leticia_mww.json` serão baixados automaticamente

> **Tempo total estimado:** ~1-2 horas

---

## Etapa 1a: Instalação das dependências

**Instale e depois clique em "Reiniciar sessão" quando aparecer o botão.**

In [ ]:
import subprocess, sys, os

print("=" * 60)
print("  ETAPA 1a: Instalação das dependências microWakeWord")
print("=" * 60)

# Clonar microWakeWord
if not os.path.exists("microWakeWord"):
    print("\n[1a] Clonando microWakeWord...")
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/kahrendt/microWakeWord"], check=True)
    print("  ✅ microWakeWord clonado")
else:
    print("  ✅ microWakeWord já existe")

def pip_install(pkgs, desc=""):
    label = desc or pkgs[0]
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *pkgs],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  ❌ {label}\n{r.stderr[-300:]}")
    else:
        print(f"  ✅ {label}")

# Dependências na ordem correta
pip_install(["git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version"],
            "pymicro-features")
pip_install(["git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f"],
            "audio-metadata")
pip_install(["-e", "./microWakeWord"], "microwakeword (local)")
pip_install(["torch==2.4.0+cu121", "torchaudio==2.4.0+cu121",
             "--index-url", "https://download.pytorch.org/whl/cu121"],
            "torch + torchaudio (cu121)")
pip_install(["piper-phonemize-cross==1.2.1"], "piper-phonemize-cross")
pip_install(["datasets", "scipy", "tqdm", "pyyaml"], "datasets / scipy / tqdm / pyyaml")

print("\n" + "=" * 60)
print("  ⚠️  REINICIE O RUNTIME AGORA")
print("  Runtime → Reiniciar sessão (ou Ctrl+M .)")
print("  Depois execute Etapa 1b em diante")
print("=" * 60)

## Etapa 1b: Verificação do ambiente

In [ ]:
import importlib, torch, os

print("=" * 60)
print("  ETAPA 1b: Verificação do ambiente")
print("=" * 60)

all_ok = True
for pkg, attr in [("torch", "__version__"), ("torchaudio", "__version__"),
                  ("microwakeword", "__version__"), ("datasets", "__version__"),
                  ("scipy", "version"), ("pymicro_features", None)]:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, attr, "ok") if attr else "ok"
        print(f"  ✅ {pkg}: {ver}")
    except ImportError:
        print(f"  ❌ {pkg}: NÃO instalado — execute Etapa 1a e reinicie")
        all_ok = False

cuda = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda else "N/A"
print(f"\n  {'✅' if cuda else '❌'} GPU: {gpu_name if cuda else 'NÃO disponível — Ative T4 GPU'}")
if not cuda:
    all_ok = False

mww_path = os.path.exists("microWakeWord")
print(f"  {'✅' if mww_path else '❌'} microWakeWord clonado: {mww_path}")
if not mww_path:
    all_ok = False

if not all_ok:
    raise RuntimeError("Corrija os erros acima antes de continuar")
print("\n[OK] ETAPA 1b CONCLUÍDA!")

## Etapa 2: Piper TTS (pt_BR faber-medium)

Mesma voz do v21 — pt_BR-faber-medium a 22050Hz (será reamostrado para 16kHz na Etapa 3).

In [ ]:
import os, subprocess

print("=" * 60)
print("  ETAPA 2: Piper sample generator + voz pt_BR")
print("=" * 60)

# Clonar piper-sample-generator
if not os.path.exists("piper-sample-generator"):
    print("\n[2a] Clonando piper-sample-generator...")
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/rhasspy/piper-sample-generator"], check=True)
    print("  ✅ piper-sample-generator clonado")
else:
    print("  ✅ piper-sample-generator já existe")

os.makedirs("piper-sample-generator/models", exist_ok=True)

# Download voz pt_BR-faber-medium
BASE_URL = "https://huggingface.co/rhasspy/piper-voices/resolve/main/pt/pt_BR/faber/medium"
for fname in ["pt_BR-faber-medium.onnx", "pt_BR-faber-medium.onnx.json"]:
    dest = f"piper-sample-generator/models/{fname}"
    if not os.path.exists(dest):
        print(f"\n[2b] Baixando {fname}...")
        subprocess.run(["wget", "-q", "--show-progress", "-O", dest,
                        f"{BASE_URL}/{fname}"], check=True)
        print(f"  ✅ {fname} baixado")
    else:
        size_mb = os.path.getsize(dest) / 1024**2
        print(f"  ✅ {fname} ({size_mb:.1f} MB)")

print("\n[OK] ETAPA 2 CONCLUÍDA!")

## Etapa 3: Gerar amostras TTS + Resample 16kHz

**FIX 20 integrado**: Piper faber-medium gera em 22050Hz → resample automático para 16kHz.

In [ ]:
import os, subprocess, torchaudio, torch
from tqdm.auto import tqdm

print("=" * 60)
print("  ETAPA 3: Geração de amostras TTS (Letícia)")
print("=" * 60)

WORD      = "letícia"
N_SAMPLES = 1000
SAMPLES_DIR = "positive_samples"
MODEL_PATH  = "piper-sample-generator/models/pt_BR-faber-medium.onnx"

os.makedirs(SAMPLES_DIR, exist_ok=True)
existing = [f for f in os.listdir(SAMPLES_DIR) if f.endswith(".wav")]

if len(existing) >= N_SAMPLES:
    print(f"  ✅ {len(existing)} amostras já existem — pulando geração")
else:
    to_generate = N_SAMPLES - len(existing)
    print(f"\n[3a] Gerando {to_generate} amostras para \"{WORD}\"...")
    cmd = [
        "python", "piper-sample-generator/generate_samples.py",
        "--model",       MODEL_PATH,
        "--text",        WORD,
        "--max-samples", str(N_SAMPLES),
        "--output-dir",  SAMPLES_DIR,
        "--cuda",
    ]
    subprocess.run(cmd, check=True)
    generated = [f for f in os.listdir(SAMPLES_DIR) if f.endswith(".wav")]
    print(f"  ✅ {len(generated)} amostras geradas")

# FIX: Piper faber-medium gera 22050Hz → microWakeWord precisa 16kHz
print("\n[FIX 20] Verificando sample rate (faber-medium = 22050Hz)...")
wavs = [f for f in os.listdir(SAMPLES_DIR) if f.endswith(".wav")]
bad = []
for f in wavs:
    try:
        info = torchaudio.info(f"{SAMPLES_DIR}/{f}")
        if info.sample_rate != 16000:
            bad.append((f, info.sample_rate))
    except Exception:
        pass

if bad:
    print(f"  ⚠️  {len(bad)} clips em {bad[0][1]}Hz — convertendo para 16kHz...")
    for fname, sr in tqdm(bad, desc="Resample 16kHz"):
        path = f"{SAMPLES_DIR}/{fname}"
        wf, _ = torchaudio.load(path)
        if wf.shape[0] > 1:
            wf = wf.mean(dim=0, keepdim=True)
        wf = torchaudio.transforms.Resample(sr, 16000)(wf)
        torchaudio.save(path, wf, 16000)
    print(f"  ✅ {len(bad)} clips convertidos")
else:
    print(f"  ✅ Todos os {len(wavs)} clips já estão em 16kHz")

# Validação final
valid = sum(1 for f in wavs if torchaudio.info(f"{SAMPLES_DIR}/{f}").sample_rate == 16000)
print(f"\n  ✅ {valid}/{len(wavs)} amostras válidas (16kHz)")
print("\n[OK] ETAPA 3 CONCLUÍDA!")

## Etapa 4: Download dos datasets negativos

Spectrogramas pré-processados do HuggingFace (kahrendt/microwakeword).
**Total ~9.7 GB** — pode levar 15-30 minutos.

In [ ]:
import os, subprocess, zipfile

print("=" * 60)
print("  ETAPA 4: Download datasets negativos")
print("=" * 60)

NEG_BASE = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main"
NEG_DATASETS = [
    ("dinner_party",      "negative_datasets/dinner_party"),
    ("dinner_party_eval", "negative_datasets/dinner_party_eval"),
    ("speech",            "negative_datasets/speech"),
    ("no_speech",         "negative_datasets/no_speech"),
]

os.makedirs("negative_datasets", exist_ok=True)

for name, dest_dir in NEG_DATASETS:
    if os.path.exists(dest_dir) and len(os.listdir(dest_dir)) > 0:
        print(f"  ✅ {name}: já existe ({len(os.listdir(dest_dir))} arquivos)")
        continue
    zip_path = f"{name}.zip"
    if not os.path.exists(zip_path):
        print(f"\n  [{name}] Baixando...")
        subprocess.run(["wget", "-q", "--show-progress", "-c",
                        f"{NEG_BASE}/{name}.zip", "-O", zip_path], check=True)
    print(f"  [{name}] Extraindo...")
    os.makedirs(dest_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest_dir)
    n = len(os.listdir(dest_dir))
    print(f"  ✅ {name}: {n} arquivos")

print("\n  Resumo:")
for _, d in NEG_DATASETS:
    n = len(os.listdir(d)) if os.path.exists(d) else 0
    print(f"    {os.path.basename(d)}: {n} arquivos")
print("\n[OK] ETAPA 4 CONCLUÍDA!")

## Etapa 5: Configuração do treinamento (YAML)

In [ ]:
import yaml

print("=" * 60)
print("  ETAPA 5: Criando training_parameters.yaml")
print("=" * 60)

config = {
    "window_step_ms": 10,
    "train_dir": "trained_models/wakeword",

    "features": [
        {
            "features_dir": "generated_augmented_features",
            "sampling_weight": 2.0,
            "penalty_weight": 1.0,
            "truth": True,
            "truncation_strategy": "truncate_start",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/speech",
            "sampling_weight": 10.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/dinner_party",
            "sampling_weight": 10.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/no_speech",
            "sampling_weight": 5.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": "negative_datasets/dinner_party_eval",
            "sampling_weight": 0.0,
            "penalty_weight": 0.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
    ],

    "training_steps":        [50000],
    "positive_class_weight": [1],
    "negative_class_weight": [20],
    "learning_rates":        [0.001],
    "batch_size":            128,

    "time_mask_max_size": [0],
    "time_mask_count":    [0],
    "freq_mask_max_size": [0],
    "freq_mask_count":    [0],

    "eval_step_interval":  1000,
    "clip_duration_ms":    1500,
    "target_minimization": 0.9,
    "minimization_metric": None,
    "maximization_metric": "average_viable_recall",
}

with open("training_parameters.yaml", "w", encoding="utf-8") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("training_parameters.yaml:")
print(yaml.dump(config, default_flow_style=False, allow_unicode=True))
print("[OK] ETAPA 5 CONCLUÍDA!")

## Etapa 6: Gerar features + Treinar modelo

**~1-2 horas** com GPU T4. O treinamento pode ser retomado se interrompido (`--restore_checkpoint 1`).

In [ ]:
import os, subprocess, sys

print("=" * 60)
print("  ETAPA 6: Feature generation + Treinamento")
print("=" * 60)

# Gerar features aumentados das amostras positivas
if not os.path.exists("generated_augmented_features") or \
   len(os.listdir("generated_augmented_features")) == 0:
    print("\n[6a] Gerando features das amostras positivas...")
    os.makedirs("generated_augmented_features", exist_ok=True)

    # Tenta os dois formatos de CLI do microWakeWord
    gen_cmds = [
        [sys.executable, "-m", "microwakeword.generate",
         "--wav_dir", "positive_samples",
         "--output_dir", "generated_augmented_features",
         "--config", "training_parameters.yaml"],
        [sys.executable, "-m", "microwakeword.generate_features",
         "--wav_dir", "positive_samples",
         "--output_dir", "generated_augmented_features",
         "--config", "training_parameters.yaml"],
        [sys.executable, "microWakeWord/microwakeword/generate.py",
         "--wav_dir", "positive_samples",
         "--output_dir", "generated_augmented_features",
         "--config", "training_parameters.yaml"],
    ]

    generated = False
    for cmd in gen_cmds:
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            print(f"  ✅ Features gerados com: {cmd[2]}")
            generated = True
            break
        else:
            print(f"  ⚠️  {cmd[2]}: {r.stderr[:200].strip()}")

    if not generated:
        print("  ❌ Não foi possível gerar features automaticamente.")
        print("  Verifique o CLI do microWakeWord:")
        subprocess.run([sys.executable, "-m", "microwakeword", "--help"],
                       capture_output=True, text=True)
        raise RuntimeError("Geração de features falhou. Veja os logs acima.")
else:
    n = len(os.listdir("generated_augmented_features"))
    print(f"  ✅ generated_augmented_features já existe ({n} arquivos)")

# Treinar
print("\n[6b] Treinando modelo microWakeWord (50k steps)...")
print("  Isso pode levar 1-2 horas com T4 GPU.\n")

train_cmd = [
    sys.executable, "-m", "microwakeword.model_train_eval",
    "--training_config=training_parameters.yaml",
    "--train", "1",
    "--restore_checkpoint", "1",
    "--test_tf_nonstreaming", "0",
    "--test_tflite_nonstreaming", "0",
    "--test_tflite_nonstreaming_quantized", "0",
    "--test_tflite_streaming", "0",
    "--test_tflite_streaming_quantized", "1",
    "--use_weights", "best_weights",
    "mixednet",
    "--pointwise_filters", "64,64,64,64",
    "--repeat_in_block", "1, 1, 1, 1",
    "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]",
    "--residual_connection", "0,0,0,0",
    "--first_conv_filters", "32",
    "--first_conv_kernel_size", "5",
    "--stride", "3",
]

subprocess.run(train_cmd, check=True)

# Verificar output
TFLITE_PATH = "trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"

if not os.path.exists(TFLITE_PATH):
    print("\n⚠️  Procurando tflite em trained_models/...")
    for root, _, files in os.walk("trained_models"):
        for f in files:
            if f.endswith(".tflite"):
                TFLITE_PATH = os.path.join(root, f)
                print(f"  Encontrado: {TFLITE_PATH}")

if os.path.exists(TFLITE_PATH):
    size_kb = os.path.getsize(TFLITE_PATH) / 1024
    print(f"\n  ✅ Modelo: {TFLITE_PATH} ({size_kb:.1f} KB)")
else:
    raise FileNotFoundError("Modelo .tflite não encontrado. Veja os logs acima.")

print("\n[OK] ETAPA 6 CONCLUÍDA!")

## Etapa 7: Empacotar, criar JSON e baixar

Gera o JSON manifest para ESPHome e baixa os dois arquivos.

In [ ]:
import os, json, shutil, glob
from google.colab import files

print("=" * 60)
print("  ETAPA 7: Empacotamento e download")
print("=" * 60)

# Localizar tflite
tflite_src = "trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
if not os.path.exists(tflite_src):
    candidates = glob.glob("trained_models/**/*.tflite", recursive=True)
    if candidates:
        tflite_src = sorted(candidates, key=os.path.getmtime)[-1]
        print(f"  Usando: {tflite_src}")
    else:
        raise FileNotFoundError("Nenhum .tflite encontrado. Execute Etapa 6 primeiro.")

os.makedirs("output", exist_ok=True)
tflite_dest = "output/leticia_mww.tflite"
shutil.copy(tflite_src, tflite_dest)

size_kb = os.path.getsize(tflite_dest) / 1024
print(f"  ✅ {tflite_dest} ({size_kb:.1f} KB)")

# JSON manifest para ESPHome micro_wake_word
# IMPORTANTE: Após subir o tflite para o GitHub Releases,
# atualize a URL abaixo antes de usar o JSON no ESPHome
GITHUB_RELEASE_URL = (
    "https://github.com/visaodeempresa/ha-wakeword-leticia"
    "/releases/download/v1.0-mww/leticia_mww.tflite"
)

manifest = {
    "type": "micro",
    "wake_word": "Letícia",
    "author": "Maycon Willian",
    "website": "https://github.com/visaodeempresa/ha-wakeword-leticia",
    "model": GITHUB_RELEASE_URL,
    "trained_languages": ["pt"],
    "version": 1,
    "micro": {
        "probability_cutoff": 0.5,
        "feature_step_size": 10,
        "sliding_window_size": 5,
        "tensor_arena_size": 26080,
        "minimum_esphome_version": "2024.7.0",
    },
}

json_dest = "output/leticia_mww.json"
with open(json_dest, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(f"  ✅ {json_dest}")

print("""
╔══════════════════════════════════════════════════════════╗
║         PRÓXIMOS PASSOS — HA Voice PE                   ║
╚══════════════════════════════════════════════════════════╝

1. CRIAR RELEASE no GitHub:
   https://github.com/visaodeempresa/ha-wakeword-leticia/releases/new
   Tag: v1.0-mww
   Upload: leticia_mww.tflite
   Copie a URL pública do arquivo

2. HOSPEDAR o JSON:
   Edite leticia_mww.json → atualize "model": com a URL do Release
   Suba leticia_mww.json também no Release
   Copie a URL pública do JSON

3. EDITAR ESPHome do HA Voice PE:
   Configurações → ESPHome → (dispositivo) → Editar
   Adicione em micro_wake_word → models:

   - model: https://URL_DO_JSON/leticia_mww.json
     id: leticia

4. INSTALAR via OTA
   (HA empurra automaticamente para o dispositivo)

5. No HA Voice PE: Wake word → Letícia ✅
""")

# Download automático
print("Baixando arquivos...")
files.download(tflite_dest)
files.download(json_dest)
print("\n[OK] ETAPA 7 CONCLUÍDA!")